PILOT SKELETON - NOT RUN - synthetic data only - no result may be quoted from this notebook

# P2 pilot 01 - environment and data manifest

Preparation for the Paper 2 (P2) pilot. Written by an agent through `build_notebooks.py`. Never executed. Not audited.

Run in Colab, this notebook:

1. mounts Drive and sets the project folder as one variable, `PROJECT_DIR`;
2. reports Python, CPU, RAM, GPU and library versions of the runtime (it reads only and installs nothing);
3. holds the data manifest: dataset, licence, source address, status;
4. defines helpers that record sha256 and write a split manifest by recording, never by frame;
5. marks the place where a download cell would go. It contains no download code.

Rules: public data only. A licence is recorded before any download. Splits are by recording or route. Times come from the runtime clock. No file is overwritten. Every printed value is PILOT. To keep this skeleton as written, run a copy (File, Save a copy in Drive).

## 1. Drive mount and project path

`PROJECT_DIR` is the one variable every other path is built from. Its default was derived from the Drive for desktop path of the project folder and was not checked in Colab: if the cell prints `project folder found: False`, edit it. Third-party data goes to `DATA_DIR` on the Colab disk, not into the shared Drive folder.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")   # the owner approves the Drive prompt in the browser

# The one variable every other path is built from.
# ASSUMPTION: derived from the Drive for desktop path of the project folder; not checked in Colab.
PROJECT_DIR = "/content/drive/MyDrive/US/Documents/Projects/PhD_Thesis_Research_UDC_CS/Opencode_PhD_Thesis_Research_UDC_CS"

NB_DIR = os.path.join(PROJECT_DIR, "exp", "P2_nb")
OUT_DIR = os.path.join(NB_DIR, "manifests")   # small text records written by this notebook
DATA_DIR = "/content/data"                    # Colab disk, not Drive: third-party data stays out of the shared folder
SAVE_OUTPUTS = False                          # True lets later cells write new files under OUT_DIR
SEED = 20261119                               # one fixed seed for the split helper

PROJECT_OK = os.path.isfile(os.path.join(NB_DIR, "README.md"))
print("in Colab:", IN_COLAB)
print("PROJECT_DIR:", PROJECT_DIR)
print("project folder found:", PROJECT_OK)
if not PROJECT_OK:
    print("Edit PROJECT_DIR: it must be the folder that holds exp/P2_nb/README.md. Nothing is saved until it is found.")

## 2. Environment report

Python, CPU, RAM, GPU, free disk and library versions, read from the runtime. A library that is absent is reported as `not installed`; nothing is installed. A runtime without a GPU is reported as such: the CPU-first route does not need one.

In [ ]:
import importlib
import importlib.metadata
import json
import platform
import shutil
import subprocess
from datetime import datetime, timezone

# Reported only. None of these is installed by this notebook.
LIBRARIES = ["numpy", "opencv-python", "opencv-python-headless", "opencv-contrib-python", "scipy",
             "scikit-learn", "pandas", "torch", "torchvision", "onnxruntime"]


def utc_now(compact=False):
    """UTC time read from the runtime clock at call time (never typed by hand)."""
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ" if compact else "%Y-%m-%dT%H:%M:%SZ")


def new_file(path):
    """Open a new text file for writing. An existing file is never replaced: the call fails instead."""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    return open(path, "x", encoding="utf-8", newline="")


def save_record(stem, record):
    """Write record as JSON to a new file OUT_DIR/<stem>_<UTC>.json. Skipped unless SAVE_OUTPUTS and PROJECT_OK."""
    if not (SAVE_OUTPUTS and PROJECT_OK):
        print("not saved (SAVE_OUTPUTS is False or the project folder was not found):", stem)
        return None
    path = os.path.join(OUT_DIR, stem + "_" + utc_now(compact=True) + ".json")
    with new_file(path) as handle:
        json.dump(record, handle, indent=1)
    print("saved:", path)
    return path


def proc_value(path, key):
    """Value of the first 'key: value' line of a /proc text file; empty text when the file or the key is absent."""
    try:
        with open(path, "r", encoding="utf-8", errors="replace") as handle:
            for line in handle:
                if line.lower().startswith(key) and ":" in line:
                    return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return ""


def environment_report():
    """Python, CPU, RAM, GPU, free disk and library versions of this runtime. Reads only; installs nothing."""
    report = {"read_utc": utc_now(), "in_colab": IN_COLAB, "python": sys.version.split()[0],
              "platform": platform.platform(),
              "cpu_model": proc_value("/proc/cpuinfo", "model name") or platform.processor() or "unknown",
              "cpu_logical_cores": os.cpu_count(),
              "ram_total": proc_value("/proc/meminfo", "memtotal") or "unknown",
              "ram_available": proc_value("/proc/meminfo", "memavailable") or "unknown",
              "disk_free_gib": round(shutil.disk_usage(os.getcwd()).free / 2 ** 30, 1)}
    smi = shutil.which("nvidia-smi")
    if smi is None:
        report["gpu"] = "none visible (no nvidia-smi on this runtime)"
    else:
        try:
            done = subprocess.run([smi, "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                                  capture_output=True, text=True, timeout=30)
            report["gpu"] = done.stdout.strip() or "nvidia-smi listed no GPU"
        except (OSError, subprocess.SubprocessError) as error:
            report["gpu"] = "nvidia-smi failed: " + type(error).__name__
    libraries = {}
    for name in LIBRARIES:
        try:
            libraries[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            libraries[name] = "not installed"
    for module in ("numpy", "cv2"):
        try:
            libraries["import " + module] = str(getattr(importlib.import_module(module), "__version__", "unknown"))
        except ImportError:
            libraries["import " + module] = "import failed"
    report["libraries"] = libraries
    return report


ENV = environment_report()
print("PILOT environment report (this runtime only)")
for key, value in ENV.items():
    print(f"  {key}: {value}")
save_record("env_report", ENV)

## 3. Data manifest

No dataset is chosen and none is downloaded. The names are the labels on the owner's deck and design slides: leads, not confirmed facts. Every licence cell reads "to confirm from the official page". Every source address cell is empty on purpose: the writer of this notebook checked no address. Task LIT_X2 collects licences and addresses (planned output: `notes/P2_LIT_X2_notes.md`). Copy from it only after it has been audited; the owner confirms each licence on the official page.

| dataset | licence | source address | status |
|---|---|---|---|
| BDD100K (video) | to confirm from the official page |  | candidate; named on the deck plan and on design slide 05; not selected; not downloaded |
| PVDN | to confirm from the official page |  | candidate; named on the deck plan; not selected; not downloaded |
| ACDC | to confirm from the official page |  | candidate; named on the deck plan; not selected; not downloaded |
| Lost and Found | to confirm from the official page |  | candidate; named on the deck plan; not selected; not downloaded |
| low-light dataset (no name given on the deck) | to confirm from the official page |  | candidate; dataset not yet named; not selected; not downloaded |
| CODA | to confirm from the official page |  | candidate; named on the deck plan; not selected; not downloaded |
| LISA (label on design slide 02) | to confirm from the official page |  | candidate; named on design slide 02; not selected; not downloaded |
| GTSRB | to confirm from the official page |  | candidate; named on design slide 02; not selected; not downloaded |

Design slide 01 also names "APDE patch attacks". The acronym is not identified (UNVERIFIED), so it has no row.

In [ ]:
LICENCE_PENDING = "to confirm from the official page"

# One row per candidate dataset. The names are labels from the owner's deck and design slides (leads).
# source_address stays empty until an audited table or the owner supplies it: never type one from memory.
DATA_MANIFEST = [
    {"dataset": "BDD100K (video)", "licence": LICENCE_PENDING, "source_address": "", "status": "candidate; named on the deck plan and on design slide 05; not selected; not downloaded"},
    {"dataset": "PVDN", "licence": LICENCE_PENDING, "source_address": "", "status": "candidate; named on the deck plan; not selected; not downloaded"},
    {"dataset": "ACDC", "licence": LICENCE_PENDING, "source_address": "", "status": "candidate; named on the deck plan; not selected; not downloaded"},
    {"dataset": "Lost and Found", "licence": LICENCE_PENDING, "source_address": "", "status": "candidate; named on the deck plan; not selected; not downloaded"},
    {"dataset": "low-light dataset (no name given on the deck)", "licence": LICENCE_PENDING, "source_address": "", "status": "candidate; dataset not yet named; not selected; not downloaded"},
    {"dataset": "CODA", "licence": LICENCE_PENDING, "source_address": "", "status": "candidate; named on the deck plan; not selected; not downloaded"},
    {"dataset": "LISA (label on design slide 02)", "licence": LICENCE_PENDING, "source_address": "", "status": "candidate; named on design slide 02; not selected; not downloaded"},
    {"dataset": "GTSRB", "licence": LICENCE_PENDING, "source_address": "", "status": "candidate; named on design slide 02; not selected; not downloaded"},
]


def ready_for_download(row):
    """True only when the licence cell holds the confirmed licence, an address is recorded and the status starts with 'owner confirmed'."""
    return (row["licence"] != LICENCE_PENDING and row["source_address"] != ""
            and row["status"].startswith("owner confirmed"))


def print_manifest(rows):
    """Print the manifest as a plain text table."""
    columns = ["dataset", "licence", "source_address", "status"]
    widths = [max(len(c), max(len(r[c]) for r in rows)) for c in columns]
    print(" | ".join(c.ljust(w) for c, w in zip(columns, widths)))
    for r in rows:
        print(" | ".join(r[c].ljust(w) for c, w in zip(columns, widths)))


print_manifest(DATA_MANIFEST)
print("licence pending:", sum(1 for r in DATA_MANIFEST if r["licence"] == LICENCE_PENDING), "of", len(DATA_MANIFEST), "rows")
save_record("data_manifest", DATA_MANIFEST)

## 4. Helpers: sha256 records and a split manifest by recording

- `write_hash_manifest` pins a folder of files: one sha256 per file plus a record with the dataset name, version, source address and the UTC of hashing.
- `split_by_recording` assigns whole recordings to splits and `write_split_manifest` writes one row per recording. A frame has no row of its own: it takes the split of its recording, so neighbouring frames of one recording cannot be separated.
- The order of recordings comes from the sha256 of seed and recording id, so it does not depend on the Python version or on hash randomisation.

In [ ]:
import csv
import hashlib

# Placeholder split names and fractions. An internal analysis plan fixes them before any run.
SPLIT_FRACTIONS = (("train", 0.5), ("val", 0.15), ("calib", 0.15), ("test", 0.2))


def sha256_file(path, chunk_bytes=1 << 20):
    """sha256 hex digest of one file, read in chunks."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk_bytes), b""):
            digest.update(block)
    return digest.hexdigest()


def write_tsv(path, header, rows):
    """Write rows to a new tab-separated file (an existing file is never replaced). Returns its sha256."""
    with new_file(path) as handle:
        writer = csv.writer(handle, delimiter="\t", lineterminator="\n")
        writer.writerow(header)
        writer.writerows(rows)
    return sha256_file(path)


def write_hash_manifest(root_dir, out_tsv, dataset="", version="", source_address=""):
    """Record the sha256 and size of every file under root_dir. Keep out_tsv outside root_dir.

    Writes out_tsv (relative_path, bytes, sha256; sorted) and out_tsv + ".json" with the dataset
    name, version, source address, UTC of hashing, file count, byte total and sha256 of out_tsv.
    """
    if not os.path.isdir(root_dir):
        raise FileNotFoundError(root_dir)
    if os.path.exists(out_tsv) or os.path.exists(out_tsv + ".json"):
        raise FileExistsError(out_tsv)   # checked before hashing, so no time is lost on a name that is taken
    rows = []
    for folder, _, files in os.walk(root_dir):
        for name in files:
            full = os.path.join(folder, name)
            relative = os.path.relpath(full, root_dir).replace(os.sep, "/")
            rows.append((relative, os.path.getsize(full), sha256_file(full)))
    rows.sort()
    record = {"dataset": dataset, "version": version, "source_address": source_address, "hashed_utc": utc_now(),
              "files": len(rows), "bytes": sum(row[1] for row in rows),
              "manifest_sha256": write_tsv(out_tsv, ["relative_path", "bytes", "sha256"], rows)}
    with new_file(out_tsv + ".json") as handle:
        json.dump(record, handle, indent=1)
    return record


def recording_of(relative_path, depth=1):
    """Recording id of a frame: its first `depth` folder names.

    Layout assumption (one folder per recording). Replace it with the dataset's own recording or
    route key, read from the dataset documentation, before any real split is written.
    """
    parts = [part for part in relative_path.replace("\\", "/").split("/") if part not in ("", ".")]
    if len(parts) <= depth:
        raise ValueError("frame is not inside a recording folder: " + relative_path)
    return "/".join(parts[:depth])


def split_by_recording(frame_paths, fractions=SPLIT_FRACTIONS, seed=SEED, key=recording_of):
    """Assign whole recordings to splits. The unit is the recording, never the frame.

    Recordings are ordered by sha256 of "seed:recording id" and cut at the cumulative fractions.
    Returns (split of each recording, frame count of each recording). With few recordings a
    split can stay empty; that is printed as a warning, not repaired.
    """
    if abs(sum(fraction for _, fraction in fractions) - 1.0) > 1e-9:
        raise ValueError("fractions must sum to 1")
    frames = {}
    for path in frame_paths:
        rec = key(path)
        frames[rec] = frames.get(rec, 0) + 1
    order = sorted(frames, key=lambda rec: hashlib.sha256(f"{seed}:{rec}".encode("utf-8")).hexdigest())
    split, start, cumulative = {}, 0, 0.0
    for index, (name, fraction) in enumerate(fractions):
        cumulative += fraction
        end = len(order) if index == len(fractions) - 1 else int(round(cumulative * len(order)))
        for rec in order[start:end]:
            split[rec] = name
        start = max(start, end)
    empty = [name for name, _ in fractions if name not in split.values()]
    if empty:
        print("WARNING: splits without any recording:", empty)
    return split, frames


def write_split_manifest(split, frames, out_tsv, fractions=SPLIT_FRACTIONS, seed=SEED):
    """Write one row per recording (recording_id, split, frames) and a JSON record beside it.

    The file has no frame rows. A frame takes the split of its recording, so neighbouring frames
    of one recording cannot land in different splits.
    """
    rows = [(rec, split[rec], frames[rec]) for rec in sorted(split)]
    names = [name for name, _ in fractions]
    record = {"unit": "recording", "seed": seed, "fractions": [list(pair) for pair in fractions],
              "written_utc": utc_now(),
              "recordings": {n: sum(1 for row in rows if row[1] == n) for n in names},
              "frames": {n: sum(row[2] for row in rows if row[1] == n) for n in names},
              "manifest_sha256": write_tsv(out_tsv, ["recording_id", "split", "frames"], rows)}
    with new_file(out_tsv + ".json") as handle:
        json.dump(record, handle, indent=1)
    return record


def read_split_manifest(tsv_path):
    """Read a split manifest back. Fails if a recording id occurs twice (it would sit in two splits)."""
    split = {}
    with open(tsv_path, "r", encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle, delimiter="\t"):
            if row["recording_id"] in split:
                raise ValueError("recording listed twice: " + row["recording_id"])
            split[row["recording_id"]] = row["split"]
    return split


def split_of_frame(relative_path, split, key=recording_of):
    """Split of one frame, looked up through its recording only."""
    return split[key(relative_path)]

In [ ]:
# DEMO on made-up file names: no data is read and nothing is written.
demo_frames = [f"rec_{r:02d}/frame_{i:04d}.png" for r in range(10) for i in range(6)]
demo_split, demo_counts = split_by_recording(demo_frames)
demo_groups = {}
for path in demo_frames:
    demo_groups.setdefault(recording_of(path), set()).add(split_of_frame(path, demo_split))
print("DEMO (made-up names, not data) recordings per split:",
      {name: sum(1 for rec in demo_split if demo_split[rec] == name) for name, _ in SPLIT_FRACTIONS})
print("DEMO every recording sits in exactly one split:", all(len(found) == 1 for found in demo_groups.values()))

# Use on real files, once a dataset is on the Colab disk and SAVE_OUTPUTS is True (not run here):
#   stamp = utc_now(compact=True)
#   write_hash_manifest(os.path.join(DATA_DIR, "<dataset>"), os.path.join(OUT_DIR, "<dataset>_sha256_" + stamp + ".tsv"),
#                       dataset="<name>", version="<version>", source_address="<from the manifest>")
#   split, frames = split_by_recording(<relative frame paths>, key=<recording or route key of that dataset>)
#   write_split_manifest(split, frames, os.path.join(OUT_DIR, "<dataset>_split_" + stamp + ".tsv"))

## 5. Download placeholder

**No download cell exists in this notebook.** The owner confirms the licence on the official page first. Until then the cell below only prints which manifest rows are ready, which is none.

In [ ]:
# =============================================================================================
# DOWNLOAD PLACEHOLDER - INTENTIONALLY EMPTY - THIS NOTEBOOK FETCHES NOTHING
# The owner confirms the licence on the official page first. Only after that:
#   1. the owner fills licence, source_address and status ("owner confirmed ...") of that row;
#   2. a cell for that one dataset is written here, saving to DATA_DIR on the Colab disk;
#   3. write_hash_manifest(...) pins the files before any use.
# =============================================================================================
ready = [row["dataset"] for row in DATA_MANIFEST if ready_for_download(row)]
print("rows with an owner-confirmed licence and a source address:", ready if ready else "none")
print("Nothing is fetched by this notebook. The owner confirms the licence first.")

## 6. Status and what is not done

- PILOT SKELETON. Not run. Not audited. Written by the same agent that wrote the builder script.
- `PROJECT_DIR` was derived from the Drive for desktop path of the project folder and was not checked in Colab.
- No dataset is chosen, no licence is recorded, no source address is recorded, nothing is downloaded.
- `recording_of` (one folder per recording) is a layout assumption. Each dataset needs its own recording or route key, checked against its documentation.
- Split names and fractions are placeholders until an internal analysis plan, self-hashed before the first run, fixes them.
- The split helper has no stratification by condition (night, weather, attack type).
- Library versions are read, not pinned: no requirements file exists yet.
- `DATA_DIR` is on the Colab disk, which does not persist between runtimes. Where third-party data is kept between sessions is an owner decision.